# TCL Phase 2 Kaggle Runs

Trains and evaluates the Phase 2 arms, as fixed in `docs/phase2/TCL-Phase2-Protocol.md`. The 14 trainings per training dataset are split into three parts, one Kaggle session each, so that every session ends well inside Kaggle's time limit and saves its own zip:

| Part | Trainings | Needs |
|---|---|---|
| 1 | arm C with lambda 0.1, 0.5, 1.0 (seed 0); arm A seeds 0-2 (seed 0 regenerates answers as the base model) | nothing |
| 2 | arm C with the chosen lambda, seeds 1-2; arm B seeds 0-2 | `LAMBDA` from part 1's `lambda_selection.json` |
| 3 | arm D seeds 0-2 | nothing |

Each session runs two trainings at a time, one per T4 GPU (`scripts/run_phase2_part.py`).

Settings: **GPU T4 x2**, **Internet on**.

In [ ]:
REPO = "https://github.com/awabmoha/truth-calibration-layer.git"
COMMIT = "21e7874474328a749c198da7a8b07fd46f3b898a"
TRAIN_DATASET = "triviaqa"   # "triviaqa" or "squad"
PART = 1                     # 1, 2, or 3
LAMBDA = None                # part 2 only: "chosen" in part 1's lambda_selection.json
LIMIT = None                 # e.g. 20 for a quick check
print(TRAIN_DATASET, PART, LAMBDA, LIMIT)

In [ ]:
import os, subprocess, json, shutil
if not os.path.exists("truth-calibration-layer"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("truth-calibration-layer")
subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
os.chdir("tcl_experiments")
subprocess.run(["pip", "install", "-q", "peft==0.21.2"], check=True)
# Kaggle ships torchao 0.10, which recent peft rejects; this project does not use torchao.
subprocess.run(["pip", "uninstall", "-y", "-q", "torchao"], check=True)
subprocess.run(["python", "scripts/check_runtime.py"], check=True)

## Base-model answers

Phase 2 trains and tests every arm on the base model's Phase 1 answers, which are committed in `runs/phase1_kaggle/qwen/`.

In [ ]:
for ds in ["triviaqa", "squad"]:
    path = f"runs/phase1_kaggle/qwen/{ds}/records.jsonl"
    with open(path) as f:
        print(ds, sum(1 for _ in f), "records")

## Runs

Per-training logs are written to `train.log` in each run directory. A failed training does not stop the others; the zip is written either way, and the last cell then reports the failure.

In [ ]:
cmd = ["python", "scripts/run_phase2_part.py", "--train-dataset", TRAIN_DATASET, "--part", str(PART),
       "--out-root", "/kaggle/working/phase2"]
if LAMBDA is not None:
    cmd += ["--lam", str(LAMBDA)]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
print(" ".join(cmd), flush=True)
part_exit = subprocess.run(cmd).returncode

## Package output

In [ ]:
zip_path = shutil.make_archive(f"/kaggle/working/phase2_{TRAIN_DATASET}_part{PART}", "zip", "/kaggle/working/phase2")
print(zip_path, os.path.getsize(zip_path) // 1_000_000, "MB")
assert part_exit == 0, f"run_phase2_part.py exited with {part_exit}; see the train.log files in the zip"